In [1]:
import json
from pathlib import Path
from typing import Any

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from datasets import Video, load_from_disk
from sklearn.model_selection import train_test_split
from tqdm import tqdm

/mnt/linuxdata/CSCI4052-Course-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths and processing config
DATASET_ROOT = Path("wlasl_full_top_100_dataset")
DATASET_DIR = DATASET_ROOT / "dataset"
LABELS_PATH = DATASET_ROOT / "labels_top100.json"

PROCESSED_ROOT = Path("processed/wlasl_top100")
CLIPS_DIR = PROCESSED_ROOT / "clips"
TMP_VIDEO_DIR = PROCESSED_ROOT / "_tmp"
METADATA_PATH = PROCESSED_ROOT / "metadata.jsonl"

NUM_FRAMES = 16
IMG_SIZE = 224
MARGIN_RATIO = 0.15
MAX_SAMPLES = 0  # 0 means all samples
RANDOM_SEED = 42

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
TMP_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Missing dataset directory: {DATASET_DIR}")
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Missing label metadata: {LABELS_PATH}")

ds = load_from_disk(str(DATASET_DIR))
if "video" in ds.features:
    ds = ds.cast_column("video", Video(decode=False))

with LABELS_PATH.open("r", encoding="utf-8") as f:
    labels_obj = json.load(f)

LABELS = labels_obj["labels"] if isinstance(labels_obj, dict) and "labels" in labels_obj else labels_obj
NUM_CLASSES = len(LABELS)

print(f"Loaded dataset rows: {len(ds)}")
print(f"Num classes: {NUM_CLASSES}")
print(f"First 10 labels: {LABELS[:10]}")

Loaded dataset rows: 10000
Num classes: 100
First 10 labels: ['book', 'drink', 'deaf', 'visit', 'wait', 'water', 'wife', 'yellow', 'backpack', 'bar']


In [3]:
# MediaPipe setup
if hasattr(mp, "solutions") and hasattr(mp.solutions, "holistic"):
    mp_holistic = mp.solutions.holistic
    holistic = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
    )
    MEDIAPIPE_MODE = "holistic"
else:
    holistic = None
    MEDIAPIPE_MODE = "fallback-full-frame"

print(f"MediaPipe mode: {MEDIAPIPE_MODE}")


def resolve_video_path(video_field: Any, sample_key: str) -> Path | None:
    if isinstance(video_field, str):
        p = Path(video_field)
        return p if p.exists() else None

    if isinstance(video_field, dict):
        path = video_field.get("path")
        if path and Path(path).exists():
            return Path(path)

        data = video_field.get("bytes")
        if data:
            tmp_path = TMP_VIDEO_DIR / f"{sample_key}.mp4"
            tmp_path.write_bytes(data)
            return tmp_path

    return None


def sample_indices(total_frames: int, num_frames: int) -> np.ndarray:
    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int32)
    return np.linspace(0, max(0, total_frames - 1), num_frames, dtype=np.int32)


def compute_bbox(results: Any, width: int, height: int, margin_ratio: float) -> tuple[int, int, int, int] | None:
    points: list[tuple[float, float]] = []

    if results and results.pose_landmarks:
        points.extend((lm.x, lm.y) for lm in results.pose_landmarks.landmark)
    if results and results.left_hand_landmarks:
        points.extend((lm.x, lm.y) for lm in results.left_hand_landmarks.landmark)
    if results and results.right_hand_landmarks:
        points.extend((lm.x, lm.y) for lm in results.right_hand_landmarks.landmark)

    if not points:
        return None

    xs = np.array([p[0] for p in points], dtype=np.float32) * width
    ys = np.array([p[1] for p in points], dtype=np.float32) * height

    x1, x2 = float(xs.min()), float(xs.max())
    y1, y2 = float(ys.min()), float(ys.max())

    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    mx = bw * margin_ratio
    my = bh * margin_ratio

    x1 = int(max(0, np.floor(x1 - mx)))
    y1 = int(max(0, np.floor(y1 - my)))
    x2 = int(min(width, np.ceil(x2 + mx)))
    y2 = int(min(height, np.ceil(y2 + my)))

    if x2 <= x1 or y2 <= y1:
        return None

    return x1, y1, x2, y2

MediaPipe mode: fallback-full-frame


In [4]:
def process_video(video_path: Path) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_indices(total_frames, NUM_FRAMES)

    frames: list[np.ndarray] = []
    prev_bbox: tuple[int, int, int, int] | None = None

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame_bgr = cap.read()
        if not ok:
            continue

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        h, w = frame_rgb.shape[:2]

        results = holistic.process(frame_rgb) if holistic is not None else None
        bbox = compute_bbox(results, w, h, MARGIN_RATIO)

        if bbox is None:
            bbox = prev_bbox if prev_bbox is not None else (0, 0, w, h)
        else:
            prev_bbox = bbox

        x1, y1, x2, y2 = bbox
        crop = frame_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            crop = frame_rgb

        resized = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        frames.append(resized)

    cap.release()

    if not frames:
        return np.zeros((NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    while len(frames) < NUM_FRAMES:
        frames.append(frames[-1].copy())

    return np.stack(frames[:NUM_FRAMES], axis=0).astype(np.uint8)


# quick sanity on first sample
first_key = str(0)
first_video_path = resolve_video_path(ds[0]["video"], first_key)
if first_video_path is None:
    print("Could not resolve first sample video path.")
else:
    first_clip = process_video(first_video_path)
    print(f"Sanity clip shape: {first_clip.shape}")
    print(f"Sanity clip dtype: {first_clip.dtype}")

Sanity clip shape: (16, 224, 224, 3)
Sanity clip dtype: uint8


In [5]:
metadata_rows = []
num_to_process = len(ds) if MAX_SAMPLES == 0 else min(MAX_SAMPLES, len(ds))

for idx, sample in tqdm(enumerate(ds), total=num_to_process, desc="Processing videos"):
    if idx >= num_to_process:
        break

    sample_key = str(sample.get("sample_id", idx))
    label_idx = int(sample["label"])
    label_text = LABELS[label_idx]

    video_path = resolve_video_path(sample["video"], sample_key)
    if video_path is None:
        continue

    clip = process_video(video_path)

    label_dir = CLIPS_DIR / label_text
    label_dir.mkdir(parents=True, exist_ok=True)
    clip_path = label_dir / f"{sample_key}.npy"
    np.save(clip_path, clip)

    metadata_rows.append(
        {
            "sample_id": sample_key,
            "clip_path": str(clip_path.as_posix()),
            "label": label_text,
            "label_idx": label_idx,
        }
    )

    if video_path.parent == TMP_VIDEO_DIR and video_path.exists():
        video_path.unlink(missing_ok=True)

print(f"Processed clips: {len(metadata_rows)}")

Processing videos:  22%|██▏       | 2166/10000 [04:49<52:09,  2.50it/s][h264 @ 0x6402b00a5a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402b00a5a80] missing picture in access unit with size 10780
[h264 @ 0x6402aeac4740] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402aeac4740] Error splitting the input into NAL units.
[h264 @ 0x6402b00a5a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402b00a5a80] missing picture in access unit with size 10780
[h264 @ 0x6402aeac4740] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402aeac4740] Error splitting the input into NAL units.
[h264 @ 0x6402b00a5a80] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402b00a5a80] missing picture in access unit with size 10780
[h264 @ 0x6402aeac4740] Invalid NAL unit size (71678 > 10776).
[h264 @ 0x6402aeac4740] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x6402ae8c7240] stream 1, offset 0x2a27a7: partial file
[h264 @ 0x6402b00a5a80] Invalid NAL unit size (71678 > 10

Processed clips: 10000


In [6]:
def save_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


save_jsonl(METADATA_PATH, metadata_rows)
print(f"Saved metadata: {METADATA_PATH}")

labels_payload = {
    "num_classes": NUM_CLASSES,
    "labels": LABELS,
    "idx_to_label": {str(i): label for i, label in enumerate(LABELS)},
}
with (PROCESSED_ROOT / "labels_top100.json").open("w", encoding="utf-8") as f:
    json.dump(labels_payload, f, indent=2)

print(f"Saved labels file: {PROCESSED_ROOT / 'labels_top100.json'}")

Saved metadata: processed/wlasl_top100/metadata.jsonl
Saved labels file: processed/wlasl_top100/labels_top100.json


In [7]:
labels_for_split = [int(row["label_idx"]) for row in metadata_rows]

train_rows, temp_rows = train_test_split(
    metadata_rows,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=labels_for_split,
)

val_rows, test_rows = train_test_split(
    temp_rows,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=[int(row["label_idx"]) for row in temp_rows],
)

train_path = PROCESSED_ROOT / "train_metadata.jsonl"
val_path = PROCESSED_ROOT / "val_metadata.jsonl"
test_path = PROCESSED_ROOT / "test_metadata.jsonl"

save_jsonl(train_path, train_rows)
save_jsonl(val_path, val_rows)
save_jsonl(test_path, test_rows)

print(f"Train split: {len(train_rows)} -> {train_path}")
print(f"Val split: {len(val_rows)} -> {val_path}")
print(f"Test split: {len(test_rows)} -> {test_path}")

Train split: 7000 -> processed/wlasl_top100/train_metadata.jsonl
Val split: 1500 -> processed/wlasl_top100/val_metadata.jsonl
Test split: 1500 -> processed/wlasl_top100/test_metadata.jsonl


In [8]:
split_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "samples": [len(train_rows), len(val_rows), len(test_rows)],
    }
)

class_df = pd.DataFrame(metadata_rows)
class_counts = (
    class_df.groupby(["label_idx", "label"], as_index=False)
    .size()
    .rename(columns={"size": "videos"})
    .sort_values(["videos", "label"], ascending=[False, True])
)

split_df

,split,samples
0,train,7000
1,val,1500
2,test,1500


In [9]:
print("Top 20 labels by processed video count:")
print(class_counts.head(20).to_string(index=False))

print("\nDone. Training notebook can now use:")
print(f"- {train_path}")
print(f"- {val_path}")
print(f"- {test_path}")
print(f"- {PROCESSED_ROOT / 'labels_top100.json'}")

Top 20 labels by processed video count:
 label_idx      label  videos
        65   accident     100
        92     africa     100
        66      apple     100
         8   backpack     100
         9        bar     100
        93 basketball     100
        44        bed     100
        41     before     100
        67       bird     100
        94   birthday     100
        29      black     100
        45       blue     100
         0       book     100
        46    bowling     100
        10    brother     100
        95      brown     100
        97        but     100
        47        can     100
        96      candy     100
        11        cat     100

Done. Training notebook can now use:
- processed/wlasl_top100/train_metadata.jsonl
- processed/wlasl_top100/val_metadata.jsonl
- processed/wlasl_top100/test_metadata.jsonl
- processed/wlasl_top100/labels_top100.json
